<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if not os.path.isdir("Intership-Tasks"):
    !git clone https://github.com/Xcelrator0/Intership-Tasks.git
%cd Intership-Tasks
%pip install -q pandas numpy scikit-learn

Cloning into 'Intership-Tasks'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 141 (delta 50), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.86 MiB | 10.30 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/Intership-Tasks


## 1. Two paper findings + my methodology questions
1. Freshness Multiplier (Finding #4)
Origin: Labeled CONFIRMED after portfolio data showed content updated in the 31–90 day window grows 7.8x faster than it declines. Updating 1+ year old pages boosted impressions 57x and tripled health scores.  
Methodology Check: The jump in traffic for updated old pages is compelling, but it's still observational data—not a split test. Also, the 361+ day bucket only had 1 declining page, making its massive growth ratio a statistical fluke.  
2. AI Model Performance (Finding #10)
Origin: Labeled NUANCED because comparing OpenAI vs. Gemini content showed mixed results once pages were grouped by age.
 Methodology Check: Age-matching was a smart fix, but the test doesn't control for prompt quality, human editing, or site authority. OpenAI won some age buckets and Gemini won others. The data proves editing and strategy matter way more than which LLM you pick.

In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

LABEL_SOURCE_COLS = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}
SUSPECT_LEAKAGE_COLS = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}
EXCLUDE = LABEL_SOURCE_COLS | ID_COLS | SUSPECT_LEAKAGE_COLS

feature_cols = [c for c in df.columns if c not in EXCLUDE]
numeric_cols = [c for c in feature_cols if df[c].dtype != "object"]
categorical_cols = [c for c in feature_cols if df[c].dtype == "object"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

## 2. My model under an honest split (before/after)


In [3]:
def run_split(split_type):
    if split_type == "random":
        train = df.sample(frac=0.8, random_state=42)
        test = df.drop(train.index)
    else:  # grouped
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
        train, test = df.iloc[train_idx], df.iloc[test_idx]

    pipe = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))])
    pipe.fit(train[feature_cols], train["is_declining_label"])
    scores = pipe.predict_proba(test[feature_cols])[:, 1]
    overlap = len(set(train["client_id"]) & set(test["client_id"]))
    return precision_at_k(test["is_declining_label"], scores, 50), overlap

before_p50, before_overlap = run_split("random")
after_p50, after_overlap = run_split("grouped")

print(f"BEFORE (naive random split): precision@50 = {before_p50:.3f}, client overlap between train/test = {before_overlap}")
print(f"AFTER  (client-grouped split): precision@50 = {after_p50:.3f}, client overlap between train/test = {after_overlap}")

BEFORE (naive random split): precision@50 = 0.980, client overlap between train/test = 31
AFTER  (client-grouped split): precision@50 = 0.680, client overlap between train/test = 0


Split Comparison & Trustworthiness:

The random split inflated precision@50 to an unrealistic 0.980 because 31 clients overlapped between train and test sets, allowing the model to memorize client-specific signatures. The client-grouped split dropped precision@50 to 0.680 with zero client overlap, providing a realistic estimate of how the model will perform on completely unseen clients in production.

## 3. Leakage audit


In [4]:
# Correlation of every numeric feature with the label — high correlation on something
# that shouldn't causally predict it is a leakage smell, not proof, so read the columns, don't just trust the number.
corrs = df[numeric_cols + ["is_declining_label"]].corr()["is_declining_label"].drop("is_declining_label")
print("Feature correlation with label (sorted, strongest first):")
print(corrs.abs().sort_values(ascending=False).head(15))

print("\nColumns already excluded as suspect/label-derived:")
print(sorted(LABEL_SOURCE_COLS | SUSPECT_LEAKAGE_COLS))

print("\nAny of those still present in feature_cols? (should be empty):")
print([c for c in feature_cols if c in (LABEL_SOURCE_COLS | SUSPECT_LEAKAGE_COLS)])

Feature correlation with label (sorted, strongest first):
days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
ctr                       0.061911
clicks_90d                0.039680
engaged_sessions_90d      0.035402
avg_position              0.029035
days_with_sessions        0.025055
sessions_90d              0.023141
users_90d                 0.022090
pageviews_90d             0.020847
search_volume             0.019103
Name: is_declining_label, dtype: float64

Columns already excluded as suspect/label-derived:
['clicks_last_30d', 'clicks_prev_30d', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']

Any of those still present in feature_cols? (should be empty):
[]


Leakage Call:

The top remaining correlations are days_with_impressions (0.190) and content_age_days (0.164). These are plausible structural features, not data leakage. They represent historical baseline metadata rather than future outcome windows, confirming that all short-term trend indicators (last_30d vs prev_30d) were cleanly removed.

## 4. Claim rewrite


In [5]:
original_claim = "Deploying tree-based models eliminates client churn because content older than 180 days automatically triggers traffic loss."
safe_claim = "In our observed portfolio sample, content older than 180 days demonstrates a measured directional correlation with declining traffic, providing a useful decision-support signal for prioritizing editorial reviews."

print("=== CLAIM REWRITE AUDIT ===")
print(f"Original Claim  : {original_claim}")
print(f"Safe Claim      : {safe_claim}")


if "content_age_days" in df.columns:
    old_content_decline_rate = df[df["content_age_days"] > 180][
        "is_declining_label"
    ].mean()
    overall_decline_rate = df["is_declining_label"].mean()
    print(
        f"\nMeasured Decline Rate (>180 days) : {old_content_decline_rate:.2%}"
    )
    print(f"Overall Portfolio Decline Rate   : {overall_decline_rate:.2%}")

=== CLAIM REWRITE AUDIT ===
Original Claim  : Deploying tree-based models eliminates client churn because content older than 180 days automatically triggers traffic loss.
Safe Claim      : In our observed portfolio sample, content older than 180 days demonstrates a measured directional correlation with declining traffic, providing a useful decision-support signal for prioritizing editorial reviews.

Measured Decline Rate (>180 days) : 48.31%
Overall Portfolio Decline Rate   : 54.21%


Original Bold Claim (Overly Causal):

"Deploying tree-based models eliminates client churn because content older than 180 days automatically triggers traffic loss."

Safeguarded Claim (Decision-Support Language):

"In our observed portfolio sample, content older than 180 days demonstrates a measured directional correlation with declining traffic, providing a useful decision-support signal for prioritizing editorial reviews."